In [6]:
#This is the starting Setup and Configuration
import yfinance as yf
import pandas as pd
import numpy as np
import requests
from transformers import pipeline

# Configuration
tickers_list = ["AAPL", "MSFT", "TSLA", "JPM", "BAC"]
NEWS_API_KEY = "9624e1d8e9fe4567a2d963f01bdb80f5"

# Download data
print("Loading stock data...")
data = yf.download(tickers_list, period="1y", progress=False)
closes = data["Close"]
print("Stock data loaded.\n")

# Load sentiment model
print("Loading FinBERT model...")
sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")
print("FinBERT loaded.\n")

Loading stock data...


/var/folders/wq/dyhyhm7579zglmwfc1m15yfm0000gn/T/ipykernel_36432/2547293195.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers_list, period="1y", progress=False)


Stock data loaded.

Loading FinBERT model...


/Users/samyushrana/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


FinBERT loaded.



In [8]:
#Helper Functions Used in it
def normalize(series):
    """Convert any series to 0-100 scale"""
    return (series - series.min()) / (series.max() - series.min()) * 100

def calculate_rsi(prices, window=14):
    """Calculate Relative Strength Index"""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def get_debt_to_equity(ticker):
    """Calculate debt-to-equity from raw financial statement data"""
    try:
        info = yf.Ticker(ticker).info
        total_debt = info.get("totalDebt", None)
        book_value = info.get("bookValue", None)
        shares_outstanding = info.get("sharesOutstanding", None)

        if total_debt is None or book_value is None or shares_outstanding is None:
            return None

        total_equity = book_value * shares_outstanding
        return total_debt / total_equity

    except Exception as e:
        print(f"Error with {ticker}: {e}")
        return None

def get_sentiment_score(ticker, company_name):
    """Calculate average sentiment from recent news headlines"""
    try:
        api_url = "https://newsapi.org/v2/everything"
        params = {
            "q": f'"{company_name}"',
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 10,
            "apiKey": NEWS_API_KEY
        }
        
        response = requests.get(api_url, params=params)
        articles = response.json()["articles"]
        headlines = [a["title"] for a in articles if a["title"]]
        
        if not headlines:
            return None
        
        scores = []
        for headline in headlines:
            result = sentiment_pipeline(headline)[0]
            score = result["score"]
            if result["label"] == "negative":
                score = -score
            elif result["label"] == "neutral":
                score = 0
            scores.append(score)
        
        return sum(scores) / len(scores)
    
    except Exception as e:
        print(f"Error with {ticker}: {e}")
        return None


In [10]:
#Component 1 - Volatility Score
print("=" * 50)
print("COMPONENT 1: VOLATILITY SCORE")
print("=" * 50)

# Calculate daily returns
daily_returns = closes.pct_change()

# Calculate volatility (standard deviation)
volatility = daily_returns.std()

# Normalize to 0-100
volatility_score = normalize(volatility)

print("\nVolatility Score (0-100):")
print(volatility_score.sort_values(ascending=False))
print("\nInterpretation: Higher = more erratic price swings = higher risk\n")


COMPONENT 1: VOLATILITY SCORE

Volatility Score (0-100):
Ticker
TSLA    100.000000
MSFT     24.033488
AAPL     11.891093
JPM       2.502888
BAC       0.000000
dtype: float64

Interpretation: Higher = more erratic price swings = higher risk



In [12]:
#Component 2- Debt Risk Score
print("=" * 50)
print("COMPONENT 2: DEBT RISK SCORE")
print("=" * 50)

# Calculate debt-to-equity for all companies
debt_equity_ratios = {}
for ticker in tickers_list:
    debt_equity_ratios[ticker] = get_debt_to_equity(ticker)

de_series = pd.Series(debt_equity_ratios)

# Normalize to 0-100
debt_score = normalize(de_series)

# Handle NaN values for banks - fill with mean of available values
debt_score = debt_score.fillna(debt_score.mean())

print("\nDebt-to-Equity Score (0-100):")
print(debt_score.sort_values(ascending=False))
print("\nInterpretation: Higher = more debt relative to equity = higher risk\n")

COMPONENT 2: DEBT RISK SCORE

Debt-to-Equity Score (0-100):
AAPL    100.00000
BAC      39.91534
JPM      39.91534
MSFT     19.74602
TSLA      0.00000
dtype: float64

Interpretation: Higher = more debt relative to equity = higher risk



In [14]:
#Component 3 - Momentum Score
print("=" * 50)
print("COMPONENT 3: MOMENTUM SCORE")
print("=" * 50)

# Calculate RSI for all companies
distance_from_neutral_dict = {}
for ticker in tickers_list:
    prices = closes[ticker]
    rsi = calculate_rsi(prices)
    latest_rsi = rsi.iloc[-1]
    distance_from_neutral_dict[ticker] = abs(latest_rsi - 50)

distance_series = pd.Series(distance_from_neutral_dict)
momentum_rsi_score = normalize(distance_series)

# Calculate moving average gap
ma_gap_dict = {}
for ticker in tickers_list:
    prices = closes[ticker]
    ma_50 = prices.rolling(window=50).mean()
    ma_200 = prices.rolling(window=200).mean()
    gap = (ma_50.iloc[-1] - ma_200.iloc[-1]) / ma_200.iloc[-1] * 100
    ma_gap_dict[ticker] = gap

ma_gap_series = pd.Series(ma_gap_dict)
ma_risk_raw = -ma_gap_series
ma_score = normalize(ma_risk_raw)

# Combine RSI and MA into one momentum score
momentum_score = (momentum_rsi_score + ma_score) / 2

print("\nMomentum Score (0-100):")
print(momentum_score.sort_values(ascending=False))
print("\nInterpretation: Higher = stronger negative momentum or erratic trading = higher risk\n")

COMPONENT 3: MOMENTUM SCORE

Momentum Score (0-100):
TSLA    90.564322
MSFT    50.000000
JPM     22.390605
BAC     20.957099
AAPL    19.938296
dtype: float64

Interpretation: Higher = stronger negative momentum or erratic trading = higher risk



In [16]:
#Component 4- Sentiment Score

print("=" * 50)
print("COMPONENT 4: SENTIMENT SCORE")
print("=" * 50)

company_names = {
    "AAPL": "Apple Inc",
    "MSFT": "Microsoft",
    "TSLA": "Tesla",
    "JPM": "JPMorgan",
    "BAC": "Bank of America"
}

print("\nFetching news headlines and analyzing sentiment...")
sentiment_scores = {}
for ticker, name in company_names.items():
    print(f"  Fetching {name}...")
    sentiment_scores[ticker] = get_sentiment_score(ticker, name)

sentiment_series = pd.Series(sentiment_scores)

# Flip sign so negative sentiment becomes high risk
sentiment_risk_raw = -sentiment_series

# Normalize to 0-100
sentiment_score = normalize(sentiment_risk_raw)

print("\nSentiment Score (0-100):")
print(sentiment_score.sort_values(ascending=False))
print("\nInterpretation: Higher = more negative news sentiment = higher risk\n")

COMPONENT 4: SENTIMENT SCORE

Fetching news headlines and analyzing sentiment...
  Fetching Apple Inc...
  Fetching Microsoft...
  Fetching Tesla...
  Fetching JPMorgan...
  Fetching Bank of America...

Sentiment Score (0-100):
TSLA    100.000000
JPM      57.063262
MSFT     45.381014
AAPL      5.328984
BAC       0.000000
dtype: float64

Interpretation: Higher = more negative news sentiment = higher risk



In [18]:
#Final Composite Risk Score

print("=" * 50)
print("FINAL COMPOSITE RISK SCORE")
print("=" * 50)

# Combine all 4 components with equal weight (25% each)
composite_risk_score = (
    volatility_score * 0.25 +
    debt_score * 0.25 +
    momentum_score * 0.25 +
    sentiment_score * 0.25
)

composite_risk_score = composite_risk_score.sort_values(ascending=False)

print("\nComposite Risk Score (0-100):")
print(composite_risk_score)
print("\nInterpretation: Higher = riskier investment based on all factors combined")
print("\nComponent Breakdown:")
print(f"  • Volatility (25%): Price stability")
print(f"  • Debt Risk (25%): Financial leverage")
print(f"  • Momentum (25%): Price trend direction")
print(f"  • Sentiment (25%): Market news sentiment\n")

FINAL COMPOSITE RISK SCORE

Composite Risk Score (0-100):
TSLA    72.641081
MSFT    34.790130
AAPL    34.289593
JPM     30.468024
BAC     15.218110
dtype: float64

Interpretation: Higher = riskier investment based on all factors combined

Component Breakdown:
  • Volatility (25%): Price stability
  • Debt Risk (25%): Financial leverage
  • Momentum (25%): Price trend direction
  • Sentiment (25%): Market news sentiment

